In [0]:
%run "../0_common/01. env-config"

In [0]:
%run "../0_common/02.bronze-helpers"

In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
source_file = f"{landing_folder_path}/{v_batch_id}/drivers.json"
table_name = f"{catalog_name}.{bronze_schema}.drivers"

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DateType

name_schema = StructType([
    StructField('givenName', StringType()),
    StructField('familyName', StringType())
])

drivers_schema = StructType([
    StructField('driverId', StringType()),
    StructField('name', name_schema),
    StructField('dateOfBirth', DateType()),
    StructField('nationality', StringType()),
    StructField('url', StringType())
])

In [0]:
drivers_df = (
    spark.read
       .format('json')
       .schema(drivers_schema)
       .option('mode', 'FAILFAST')
       .load(source_file)
)

In [0]:
drivers_df = add_metadata_columns(drivers_df)

display(drivers_df.head(5))

In [0]:
write_to_bronze(
    df=drivers_df,
    target_table=table_name,
    batch_id=v_batch_id
)

In [0]:
display(spark.table(table_name))